# SHAP Analysis For Telecom Customer Churn

            This notebook reproduces the explainability figures for the best-performing Phase 1 churn model.
            Each section is written to help non-technical readers understand which customer traits pushed the
            model toward a churn or retention prediction.


In [ ]:
from pathlib import Path
            import sys

            def find_project_root(start: Path) -> Path:
                current = start.resolve()
                for candidate in [current, *current.parents]:
                    if (candidate / "src").exists() and (candidate / "params.yaml").exists():
                        return candidate
                raise RuntimeError("Could not find project root.")

            PROJECT_ROOT = find_project_root(Path.cwd())
            if str(PROJECT_ROOT) not in sys.path:
                sys.path.insert(0, str(PROJECT_ROOT))

            import pandas as pd
            from IPython.display import Image, Markdown, display
            from src.explainability.shap_analysis import run_shap_analysis, slugify

            analysis = run_shap_analysis(config_path=PROJECT_ROOT / "params.yaml")
            top_features = pd.read_csv(PROJECT_ROOT / "results" / "top_features.csv")
            top_features.head(10)


## Global Feature Importance

            The two plots below answer complementary questions. The summary bar plot shows which variables
            matter most overall, while the beeswarm plot shows whether the same feature pushes different
            customers in different directions.


In [ ]:
display(Markdown("### SHAP Summary Bar Plot"))
            display(Image(filename=str(PROJECT_ROOT / "plots" / "shap_summary_bar.png")))

            display(Markdown("### SHAP Beeswarm Plot"))
            display(Image(filename=str(PROJECT_ROOT / "plots" / "shap_beeswarm.png")))


## Dependence Plots

            These charts focus on the five most important drivers one at a time. Each point is a customer.
            The horizontal position shows the feature value, the vertical position shows the SHAP effect on churn risk,
            and the color reflects the strongest correlated companion feature.


In [ ]:
dependence_features = top_features["feature"].head(5).tolist()
            for feature_name in dependence_features:
                display(Markdown(f"### {feature_name.replace('_', ' ').title()}"))
                display(Image(filename=str(PROJECT_ROOT / "plots" / f"shap_dependence_{slugify(feature_name)}.png")))


## Local Customer Explanations

            The waterfall plots below explain individual predictions. Features pushing the customer toward
            churn appear on one side of the baseline prediction, while protective features push the score back down.


In [ ]:
for index in range(1, 6):
                display(Markdown(f"### Sample {index}"))
                display(Image(filename=str(PROJECT_ROOT / "plots" / f"shap_waterfall_sample_{index}.png")))


## Business Interpretation

            The final markdown report translates the model findings into operational language for churn-reduction planning.


In [ ]:
display(Markdown((PROJECT_ROOT / "paper" / "churn_drivers_analysis.md").read_text(encoding="utf-8")))
